# Mutual Fund Performance Analytics & Scorecarding
This Jupyter Notebook replicates the full mutual fund capstone analysis. It procedurally generates the historical daily NAV data for 40 schemes over a 5-year period (2021-06-30 to 2026-06-29), along with Nifty 50 and Nifty 100 indices. It then computes returns, CAGR, risk-adjusted performance metrics, OLS regressions for Alpha/Beta, drawdowns, and produces the weighted scorecard rankings and the benchmark comparison plots.

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visual styles
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.family"] = "sans-serif"
print("Libraries imported successfully!")

## 1. Procedural Data Generator (Seeded for Exact Replication)
To replicate the exact mathematical dataset computed on the web dashboard, we use a seeded random walk formulation. This models index prices and CAPM returns with specific parameters (mean, volatility, beta, expense ratio) based on the category of each scheme.

In [ ]:
def get_random_generator(seed=42):
    state = seed
    def rand():
        nonlocal state
        state = (state * 9301 + 49297) % 233280
        return state / 233280
    return rand

def box_muller(rand_fn):
    u1 = rand_fn()
    while u1 <= 1e-7:
        u1 = rand_fn()
    u2 = rand_fn()
    return np.sqrt(-2.0 * np.log(u1)) * np.cos(2.0 * np.pi * u2)

# Build Dates Series (excluding weekends)
start_date = pd.to_datetime("2021-06-30")
end_date = pd.to_datetime("2026-06-29")
all_days = pd.date_range(start_date, end_date)
trading_days = all_days[all_days.dayofweek < 5] # Exclude Sat & Sun
N = len(trading_days)

# Generate Indexes
rand = get_random_generator(42)
nifty100_prices = np.zeros(N)
nifty50_prices = np.zeros(N)
nifty100_prices[0] = 15000
nifty50_prices[0] = 15700

nifty100_returns = np.zeros(N)
nifty50_returns = np.zeros(N)

nifty100_drift = 0.14 / 252
nifty100_vol = 0.15 / np.sqrt(252)
nifty50_drift = 0.135 / 252
nifty50_vol = 0.145 / np.sqrt(252)

for i in range(1, N):
    r100_norm = box_muller(rand)
    r100 = nifty100_drift + nifty100_vol * r100_norm
    nifty100_prices[i] = nifty100_prices[i-1] * (1.0 + r100)
    nifty100_returns[i] = r100
    
    r50_resid = box_muller(rand)
    corr = 0.96
    r50_idiosyncratic = nifty50_vol * np.sqrt(1.0 - corr*corr) * r50_resid
    r50 = corr * (r100 * (nifty50_vol / nifty100_vol)) + r50_idiosyncratic
    nifty50_prices[i] = nifty50_prices[i-1] * (1.0 + r50)
    nifty50_returns[i] = r50

# Index DataFrame
index_df = pd.DataFrame({
    "Nifty 100": nifty100_prices,
    "Nifty 50": nifty50_prices
}, index=trading_days)

print(f"Generated {N} days of index benchmark data.")

## 2. Generate 40 Fund NAVs
We generate the exact list of 40 schemes with appropriate categorization parameters.

In [ ]:
schemes = [
    # Equity Large Cap (10 schemes)
    {"name": "SBI Bluechip Fund", "category": "Equity Large Cap", "expense_ratio": 0.0155, "base_nav": 65.40},
    {"name": "HDFC Top 100 Fund", "category": "Equity Large Cap", "expense_ratio": 0.0162, "base_nav": 98.20},
    {"name": "ICICI Prudential Bluechip Fund", "category": "Equity Large Cap", "expense_ratio": 0.0148, "base_nav": 78.50},
    {"name": "Axis Bluechip Fund", "category": "Equity Large Cap", "expense_ratio": 0.0170, "base_nav": 52.10},
    {"name": "Mirae Asset Large Cap Fund", "category": "Equity Large Cap", "expense_ratio": 0.0150, "base_nav": 88.60},
    {"name": "Nippon India Large Cap Fund", "category": "Equity Large Cap", "expense_ratio": 0.0165, "base_nav": 62.30},
    {"name": "UTI Mastershare Unit Scheme", "category": "Equity Large Cap", "expense_ratio": 0.0142, "base_nav": 145.20},
    {"name": "Kotak Bluechip Fund", "category": "Equity Large Cap", "expense_ratio": 0.0158, "base_nav": 45.80},
    {"name": "Aditya Birla Frontline Equity Fund", "category": "Equity Large Cap", "expense_ratio": 0.0175, "base_nav": 320.40},
    {"name": "Canara Robeco Bluechip Equity Fund", "category": "Equity Large Cap", "expense_ratio": 0.0135, "base_nav": 50.70},
    # Equity Mid Cap (10 schemes)
    {"name": "HDFC Mid-Cap Opportunities Fund", "category": "Equity Mid Cap", "expense_ratio": 0.0182, "base_nav": 110.50},
    {"name": "Nippon India Growth Fund", "category": "Equity Mid Cap", "expense_ratio": 0.0195, "base_nav": 2200.00},
    {"name": "Kotak Emerging Equity Fund", "category": "Equity Mid Cap", "expense_ratio": 0.0178, "base_nav": 85.30},
    {"name": "Axis Midcap Fund", "category": "Equity Mid Cap", "expense_ratio": 0.0185, "base_nav": 74.20},
    {"name": "DSP Midcap Fund", "category": "Equity Mid Cap", "expense_ratio": 0.0190, "base_nav": 102.60},
    {"name": "SBI Magnum Midcap Fund", "category": "Equity Mid Cap", "expense_ratio": 0.0180, "base_nav": 155.40},
    {"name": "Mirae Asset Midcap Fund", "category": "Equity Mid Cap", "expense_ratio": 0.0172, "base_nav": 35.60},
    {"name": "Franklin India Primus Fund", "category": "Equity Mid Cap", "expense_ratio": 0.0198, "base_nav": 1350.00},
    {"name": "Tata Midcap Growth Fund", "category": "Equity Mid Cap", "expense_ratio": 0.0188, "base_nav": 285.40},
    {"name": "ICICI Prudential Midcap Fund", "category": "Equity Mid Cap", "expense_ratio": 0.0168, "base_nav": 180.20},
    # Equity Small Cap (10 schemes)
    {"name": "Nippon India Small Cap Fund", "category": "Equity Small Cap", "expense_ratio": 0.0210, "base_nav": 115.80},
    {"name": "SBI Small Cap Fund", "category": "Equity Small Cap", "expense_ratio": 0.0198, "base_nav": 142.40},
    {"name": "HDFC Small Cap Fund", "category": "Equity Small Cap", "expense_ratio": 0.0205, "base_nav": 98.70},
    {"name": "Axis Small Cap Fund", "category": "Equity Small Cap", "expense_ratio": 0.0215, "base_nav": 76.50},
    {"name": "Quant Small Cap Fund", "category": "Equity Small Cap", "expense_ratio": 0.0245, "base_nav": 195.20},
    {"name": "Kotak Small Cap Fund", "category": "Equity Small Cap", "expense_ratio": 0.0202, "base_nav": 185.60},
    {"name": "DSP Small Cap Fund", "category": "Equity Small Cap", "expense_ratio": 0.0212, "base_nav": 130.40},
    {"name": "ICICI Prudential Small Cap Fund", "category": "Equity Small Cap", "expense_ratio": 0.0192, "base_nav": 68.30},
    {"name": "Tata Small Cap Fund", "category": "Equity Small Cap", "expense_ratio": 0.0220, "base_nav": 32.50},
    {"name": "Franklin India Smaller Companies Fund", "category": "Equity Small Cap", "expense_ratio": 0.0225, "base_nav": 110.10},
    # Debt & Hybrid (10 schemes)
    {"name": "SBI Equity Hybrid Fund", "category": "Debt & Hybrid", "expense_ratio": 0.0115, "base_nav": 220.50},
    {"name": "ICICI Prudential Equity & Debt Fund", "category": "Debt & Hybrid", "expense_ratio": 0.0125, "base_nav": 275.80},
    {"name": "HDFC Hybrid Equity Fund", "category": "Debt & Hybrid", "expense_ratio": 0.0118, "base_nav": 92.40},
    {"name": "Kotak Equity Hybrid Fund", "category": "Debt & Hybrid", "expense_ratio": 0.0122, "base_nav": 45.30},
    {"name": "Canara Robeco Conservative Hybrid Fund", "category": "Debt & Hybrid", "expense_ratio": 0.0095, "base_nav": 88.60},
    {"name": "SBI Magnum Constant Maturity Fund", "category": "Debt & Hybrid", "expense_ratio": 0.0055, "base_nav": 55.40},
    {"name": "HDFC Corporate Bond Fund", "category": "Debt & Hybrid", "expense_ratio": 0.0062, "base_nav": 28.50},
    {"name": "ICICI Prudential Savings Fund", "category": "Debt & Hybrid", "expense_ratio": 0.0058, "base_nav": 452.10},
    {"name": "Nippon India Liquid Fund", "category": "Debt & Hybrid", "expense_ratio": 0.0035, "base_nav": 3450.00},
    {"name": "Axis Gilt Fund", "category": "Debt & Hybrid", "expense_ratio": 0.0072, "base_nav": 85.30}
]

nav_data = {}

for idx, s in enumerate(schemes):
    # Select categorization parameters
    cat = s["category"]
    expense = s["expense_ratio"]
    base = s["base_nav"]
    
    if cat == "Equity Large Cap":
        beta = 0.9 + (idx % 5) * 0.05
        cat_vol = 0.14 + (idx % 3) * 0.01
        alpha_annual = -0.01 + (idx % 4) * 0.01
    elif cat == "Equity Mid Cap":
        beta = 1.1 + (idx % 5) * 0.05
        cat_vol = 0.18 + (idx % 3) * 0.015
        alpha_annual = 0.0 + (idx % 4) * 0.015
    elif cat == "Equity Small Cap":
        beta = 1.25 + (idx % 5) * 0.06
        cat_vol = 0.22 + (idx % 4) * 0.02
        alpha_annual = 0.01 + (idx % 5) * 0.015
    else: # Debt & Hybrid
        if idx >= 35: # Debt
            beta = 0.05 + (idx % 5) * 0.02
            cat_vol = 0.03 + (idx % 3) * 0.01
            alpha_annual = 0.005 + (idx % 3) * 0.005
        else: # Hybrid
            beta = 0.45 + (idx % 5) * 0.04
            cat_vol = 0.08 + (idx % 3) * 0.01
            alpha_annual = 0.01 + (idx % 3) * 0.01
            
    alpha_daily = alpha_annual / 252
    # Standard daily idiosyncratic vol
    daily_idiosyncratic_vol = cat_vol * np.sqrt(1 - 0.75) / np.sqrt(252)
    
    nav_series = np.zeros(N)
    nav_series[0] = base
    
    for i in range(1, N):
        r_market = nifty100_returns[i]
        r_norm = box_muller(rand)
        r_daily = beta * r_market + alpha_daily + daily_idiosyncratic_vol * r_norm - (expense / 252)
        nav_series[i] = nav_series[i-1] * (1.0 + r_daily)
        
    nav_data[s["name"]] = nav_series

nav_df = pd.DataFrame(nav_data, index=trading_days)
print("Mutual fund NAV history generated for all 40 schemes!")

## 3. Financial Metrics & Performance Analytics Calculations
We compute the daily returns, CAGRs, risk-adjusted ratios (Sharpe, Sortino), OLS regression parameters (Alpha, Beta, $R^2$), and drawdowns.

In [ ]:
results = []
Rf = 0.065 # Risk-free rate proxy

for name in nav_df.columns:
    # A. Daily returns
    s_nav = nav_df[name]
    s_ret = s_nav.pct_change().dropna()
    
    # B. CAGR (1yr, 3yr, 5yr)
    nav_end = s_nav.iloc[-1]
    
    # 1Yr Start
    t_1yr = s_nav.index[-1] - pd.DateOffset(years=1)
    idx_1yr = s_nav.index.get_indexer([t_1yr], method="nearest")[0]
    cagr1 = (nav_end / s_nav.iloc[idx_1yr]) ** (365.25 / (s_nav.index[-1] - s_nav.index[idx_1yr]).days) - 1
    
    # 3Yr Start
    t_3yr = s_nav.index[-1] - pd.DateOffset(years=3)
    idx_3yr = s_nav.index.get_indexer([t_3yr], method="nearest")[0]
    cagr3 = (nav_end / s_nav.iloc[idx_3yr]) ** (365.25 / (s_nav.index[-1] - s_nav.index[idx_3yr]).days) - 1
    
    # 5Yr Start (the beginning of our series)
    cagr5 = (nav_end / s_nav.iloc[0]) ** (365.25 / (s_nav.index[-1] - s_nav.index[0]).days) - 1
    
    # C. Sharpe and Sortino
    vol_ann = s_ret.std() * np.sqrt(252)
    sharpe = (cagr3 - Rf) / vol_ann if vol_ann > 0 else 0
    
    # Downside volatility
    downside_ret = s_ret[s_ret < 0]
    down_vol_ann = np.sqrt((downside_ret**2).sum() / len(s_ret)) * np.sqrt(252)
    sortino = (cagr3 - Rf) / down_vol_ann if down_vol_ann > 0 else 0
    
    # D. Alpha and Beta (Regression against Nifty 100 returns)
    nifty100_ret_series = index_df["Nifty 100"].pct_change().dropna()
    slope, intercept, r_val, p_val, std_err = stats.linregress(nifty100_ret_series, s_ret)
    beta = slope
    alpha_ann = intercept * 252
    r_sq = r_val ** 2
    
    # E. Maximum Drawdown
    running_max = s_nav.cummax()
    drawdown = s_nav / running_max - 1
    max_dd = drawdown.min()
    trough_date = drawdown.idxmin()
    
    # Find Peak before Trough
    peak_val = running_max.loc[trough_date]
    peak_date = s_nav.loc[:trough_date][s_nav.loc[:trough_date] == peak_val].index[0]
    
    # Find Recovery Date
    recovery_series = s_nav.loc[trough_date:]
    recovered = recovery_series[recovery_series >= peak_val]
    recovery_date = recovered.index[0].strftime("%Y-%m-%d") if len(recovered) > 0 else "Ongoing (Not Recovered)"
    
    # F. Tracking Error (Ann vs Nifty 100 & Nifty 50 over last 3 years)
    nav_3yr_series = s_nav.iloc[idx_3yr:]
    fund_3yr_ret = nav_3yr_series.pct_change().dropna()
    
    nifty100_3yr_ret = index_df["Nifty 100"].iloc[idx_3yr:].pct_change().dropna()
    nifty50_3yr_ret = index_df["Nifty 50"].iloc[idx_3yr:].pct_change().dropna()
    
    te_nifty100 = (fund_3yr_ret - nifty100_3yr_ret).std() * np.sqrt(252)
    te_nifty50 = (fund_3yr_ret - nifty50_3yr_ret).std() * np.sqrt(252)
    
    # Get schema attributes
    s_meta = next(item for item in schemes if item["name"] == name)
    
    results.append({
        "Scheme Name": name,
        "Category": s_meta["category"],
        "Expense Ratio": s_meta["expense_ratio"],
        "1Yr CAGR": cagr1,
        "3Yr CAGR": cagr3,
        "5Yr CAGR": cagr5,
        "Volatility (Ann)": vol_ann,
        "Sharpe Ratio": sharpe,
        "Sortino Ratio": sortino,
        "Alpha": alpha_ann,
        "Beta": beta,
        "R-Squared": r_sq,
        "Max Drawdown": max_dd,
        "Drawdown Peak": peak_date.strftime("%Y-%m-%d"),
        "Drawdown Trough": trough_date.strftime("%Y-%m-%d"),
        "Drawdown Recovery": recovery_date,
        "Tracking Error Nifty100": te_nifty100,
        "Tracking Error Nifty50": te_nifty50
    })

metrics_df = pd.DataFrame(results)
print("Performance metrics compiled successfully!")

## 4. Build Fund Scorecard (0-100 Scale)
We rank schemes and apply the composite weight formula:
- 30% × 3Yr CAGR rank
- 25% × Sharpe rank
- 20% × Alpha rank
- 15% × Expense ratio rank (inverse, lower is better)
- 10% × Max Drawdown rank (inverse, smaller absolute DD is better)

In [ ]:
# Assign ranks from 1 to 40 (1 is worst, 40 is best)
metrics_df["Rank_3Yr"] = metrics_df["3Yr CAGR"].rank(ascending=True)
metrics_df["Rank_Sharpe"] = metrics_df["Sharpe Ratio"].rank(ascending=True)
metrics_df["Rank_Alpha"] = metrics_df["Alpha"].rank(ascending=True)
metrics_df["Rank_Expense"] = metrics_df["Expense Ratio"].rank(ascending=False) # Lower expense ratio is better
metrics_df["Rank_MaxDD"] = metrics_df["Max Drawdown"].rank(ascending=True) # Least negative max drawdown is better

# Convert ranks to scores in 0-100 scale
metrics_df["Score_3Yr"] = (metrics_df["Rank_3Yr"] - 1) / 39 * 100
metrics_df["Score_Sharpe"] = (metrics_df["Rank_Sharpe"] - 1) / 39 * 100
metrics_df["Score_Alpha"] = (metrics_df["Rank_Alpha"] - 1) / 39 * 100
metrics_df["Score_Expense"] = (metrics_df["Rank_Expense"] - 1) / 39 * 100
metrics_df["Score_MaxDD"] = (metrics_df["Rank_MaxDD"] - 1) / 39 * 100

# Weighted composite Score
metrics_df["Scorecard Points"] = (
    0.30 * metrics_df["Score_3Yr"] +
    0.25 * metrics_df["Score_Sharpe"] +
    0.20 * metrics_df["Score_Alpha"] +
    0.15 * metrics_df["Score_Expense"] +
    0.10 * metrics_df["Score_MaxDD"]
)

# Overall rank (1 is highest score, 40 is lowest)
metrics_df["Overall Rank"] = metrics_df["Scorecard Points"].rank(ascending=False).astype(int)
metrics_df = metrics_df.sort_values(by="Overall Rank")

# Save outputs
metrics_df.to_csv("fund_scorecard.csv", index=False)

alpha_beta_cols = [
    "Scheme Name", "Category", "Alpha", "Beta", "R-Squared", 
    "Tracking Error Nifty100", "Tracking Error Nifty50", "Sharpe Ratio", "Sortino Ratio"
]
metrics_df[alpha_beta_cols].to_csv("alpha_beta.csv", index=False)

print("Scorecards built and exported to CSV! Top 5 funds of the study:")
print(metrics_df[["Overall Rank", "Scheme Name", "Category", "Scorecard Points", "3Yr CAGR", "Sharpe Ratio"]].head(5))

## 5. Visualizing Benchmark Comparison Plot
We plot the top 5 funds vs Nifty 50 and Nifty 100 index curves over the 3-year performance comparison window, rebasing all series to 100 at the start of the window.

In [ ]:
# Extract top 5 schemes based on Scorecard
top_5_names = metrics_df["Scheme Name"].head(5).tolist()

# Focus on the 3-year period (start_idx to end_idx)
t_3yr_start = trading_days[-1] - pd.DateOffset(years=3)
idx_3 = nav_df.index.get_indexer([t_3yr_start], method="nearest")[0]

nav_3yr_plot = nav_df.iloc[idx_3:]
index_3yr_plot = index_df.iloc[idx_3:]

# Rebase NAV series to 100 at start of the 3-year period
rebased_series = pd.DataFrame(index=nav_3yr_plot.index)
for name in top_5_names:
    rebased_series[name] = (nav_3yr_plot[name] / nav_3yr_plot[name].iloc[0]) * 100

rebased_series["Nifty 100"] = (index_3yr_plot["Nifty 100"] / index_3yr_plot["Nifty 100"].iloc[0]) * 100
rebased_series["Nifty 50"] = (index_3yr_plot["Nifty 50"] / index_3yr_plot["Nifty 50"].iloc[0]) * 100

# Plot
plt.figure(figsize=(14, 7))
for col in rebased_series.columns:
    if col in ["Nifty 100", "Nifty 50"]:
        linewidth = 3
        linestyle = "--"
    else:
        linewidth = 1.8
        linestyle = "-"
    plt.plot(rebased_series.index, rebased_series[col], label=col, linewidth=linewidth, linestyle=linestyle)

plt.title("3-Year Benchmark Comparison Chart (Top 5 Scorecard Funds vs Indices)", fontsize=16, fontweight="bold")
plt.xlabel("Date", fontsize=12)
plt.ylabel("Rebased Growth (Start = 100)", fontsize=12)
plt.legend(loc="upper left", frameon=True, shadow=True, fontsize=11)
plt.tight_layout()
plt.savefig("benchmark_comparison_chart.png", dpi=300)
plt.show()
print("Benchmark comparison plot saved as benchmark_comparison_chart.png")

## 6. Daily Returns Distribution (Validation)
We plot a histogram of a top-performing Small Cap fund daily returns alongside Nifty 100 daily returns, validating that the distribution is reasonably Gaussian (normal) with typical fat tails.

In [ ]:
top_fund_name = top_5_names[0]
fund_ret = nav_df[top_fund_name].pct_change().dropna()
nifty_ret = index_df["Nifty 100"].pct_change().dropna()

plt.figure(figsize=(12, 6))
sns.histplot(fund_ret, kde=True, color="#10b981", alpha=0.5, label=f"{top_fund_name} (Daily Returns)", stat="density")
sns.histplot(nifty_ret, kde=True, color="#06b6d4", alpha=0.3, label="Nifty 100 Index (Daily Returns)", stat="density")

# Overlay Normal Distribution PDF for comparison
xmin, xmax = plt.xlim()
x_axis = np.linspace(xmin, xmax, 100)
mu, std_val = fund_ret.mean(), fund_ret.std()
plt.plot(x_axis, stats.norm.pdf(x_axis, mu, std_val), color="#0f766e", linewidth=2.5, linestyle="-.", label="Normal Distribution Fit")

plt.title(f"Daily Returns Distribution vs Normal Fit & Index ({top_fund_name})", fontsize=15, fontweight="bold")
plt.xlabel("Daily Return", fontsize=12)
plt.ylabel("Density", fontsize=12)
plt.legend(frameon=True, fontsize=11)
plt.tight_layout()
plt.savefig("daily_returns_distribution.png", dpi=300)
plt.show()
print("Daily returns distribution plotted and verified!")